## Import library

In [11]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import AnalyticValue as AV
from matplotlib import pyplot as plt
from scipy.stats import linregress
warnings.filterwarnings(action="ignore")

In [12]:
# make save directory
if not os.path.exists("_Figure"): os.mkdir("_Figure")
if not os.path.exists("_Figure/CGF/"): os.mkdir("_Figure/CGF/")
# if not os.path.exists("_Figure/LDF/"): os.mkdir("_Figure/LDF/")
if not os.path.exists("_Figure/Work/"): os.mkdir("_Figure/Work/")
if not os.path.exists("_Figure/Dynamics/"): os.mkdir("_Figure/Dynamics/")
if not os.path.exists("_Figure/Error/"): os.mkdir("_Figure/Error/")

In [13]:
lmbdas = [-0.1,-0.2,-0.3,-0.4,-0.5,-0.6,-0.7,-0.8,-0.9,-1.0,
          -1.1,-1.2,-1.3,-1.4,-1.5,-1.6,-1.7,-1.8,-1.9,-2.0]

epochs = 50000; L = 1.0
μ = 1.0; τ = 1.0; K = 10.0; Da = 2.0; kbT = 0.5

In [14]:
def f2s(val:float) -> str:
    s = format(val, ".0e")
    s = re.sub(r'e([-+]?)0*(\d+)', r'e\1\2', s)
    return s

### Train dynamics

In [15]:
def ProcessTrain(what,N,n_hid,n_block,lmbdas,lr):
    data_for_all_lmbda = []
    for lmbda in lmbdas:
        data = pd.read_pickle("data/Convention_"+what+"_N={0}_H={1}_B={2}_λ={3}_lrp={4}.pkl".format(N,n_hid,n_block,lmbda,f2s(lr)))
        data_for_all_lmbda.append(np.array(data))
    return data_for_all_lmbda

In [8]:
def PlotLoss(N,n_hid,n_block,lmbdas,lr):
    if not os.path.exists('_Figure/Dynamics/N={0}/'.format(N)): os.mkdir('_Figure/Dynamics/N={0}/'.format(N))
    if not os.path.exists('_Figure/Dynamics/N={0}/Loss/'.format(N)): os.mkdir('_Figure/Dynamics/N={0}/Loss/'.format(N))
    true = AV.CGF(N,τ,μ,K,Da,kbT,np.array(lmbdas))
    # load data
    data = ProcessTrain('SCGF',N,n_hid,n_block,lmbdas,lr)
    for n in range(len(lmbdas)):
        plt.figure(dpi=400)
        x = np.array([e for e in range(epochs)])
        y = -np.array(data[n])
        plt.plot(x+1,y,marker='o',markersize=3,c='blue')
        # 꾸미기
        plt.ylim(-1.,5.)
        plt.grid()
        plt.yticks(np.arange(-1,5.,0.25))
        plt.axhline(y=-true[n],c='black')
        plt.xlabel('Epoch'); plt.ylabel('Loss')
        if not os.path.exists('_Figure/Dynamics/N={0}/Loss/lr={1}/'.format(N,f2s(lr))): os.mkdir('_Figure/Dynamics/N={0}/Loss/lr={1}/'.format(N,f2s(lr)))
        plt.savefig("_Figure/Dynamics/N={0}/Loss/lr={1}/B={2}_λ={3}.png".format(N,f2s(lr),n_block,lmbdas[n]),bbox_inches='tight')
        plt.close()
        plt.clf()

N = 21; lr = 1e-3; n_blocks = [1]
n_hids = [25, 50]
for n_hid in n_hids:
    for n_block in n_blocks:
        PlotLoss(N,n_hid,n_block,lmbdas,lr)

<Figure size 640x480 with 0 Axes>

### CGF

In [9]:
def PlotCGF(N,n_hids,n_block,lmbdas,lr):
    if not os.path.exists("_Figure/CGF/N={0}/".format(N)): os.mkdir("_Figure/CGF/N={0}/".format(N)) 
    if not os.path.exists("_Figure/CGF/N={0}/CGF/".format(N)): os.mkdir("_Figure/CGF/N={0}/CGF/".format(N))
    # true value
    ACGF = AV.CGF(N,τ,μ,K,Da,kbT,np.array(lmbdas))
    # plot
    plt.figure(dpi=400)
    for n_hid in n_hids:
        # load data
        mean = []; std = []
        for lmbda in lmbdas:
            data = -pd.read_pickle("result/Convention_LSCGF_N={0}_H={1}_B={2}_λ={3}_lrp={4}.pkl".format(N,n_hid,n_block,lmbda,f2s(lr)))
            # compute mean and std 
            std.append(data.std())
            mean.append(data.mean())
        plt.errorbar(lmbdas, mean, std, capsize=3, fmt='o', ls='-', label='H = {0}'.format(n_hid))
    # 꾸미기
    plt.plot(np.linspace(-2.0,0.0,100000), AV.CGF(N,τ,μ,K,Da,kbT,np.linspace(-2.0,0.0,100000)),c='black')
    plt.scatter(lmbdas,ACGF,marker='o',c='black')
    plt.grid()
    plt.xlabel("λ")
    plt.ylabel("ψ(λ)")
    plt.legend()
    plt.ylim(int(2*min(ACGF)), 0.1)
    plt.yticks(np.arange(int(2*min(ACGF)),0.1,0.1))
    plt.savefig("_Figure/CGF/N={0}/CGF/lr={1}.png".format(N,f2s(lr)),bbox_inches='tight')
    plt.close()
    plt.clf()

N = 11; lr = 1e-3; n_blocks = [1]
n_hids = [25,50]
for n_hid in n_hids:
    PlotCGF(N,n_hids,n_block,lmbdas,lr)

FileNotFoundError: [Errno 2] No such file or directory: 'result/Convention_LSCGF_N=11_H=25_B=1_λ=-0.1_lrp=1e-3.pkl'

<Figure size 2560x1920 with 0 Axes>

In [17]:
def PlotCGFdiff(N,n_hid,n_blocks,lmbdas,lr):
    if not os.path.exists("_Figure/CGF/N={0}/".format(N)): os.mkdir("_Figure/CGF/N={0}/".format(N)) 
    if not os.path.exists("_Figure/CGF/N={0}/Difference/".format(N)): os.mkdir("_Figure/CGF/N={0}/Difference/".format(N))
    # true value
    ACGF = AV.CGF(N,τ,μ,K,Da,kbT,np.array(lmbdas))
    # plot
    plt.figure(dpi=400)
    for n_block in n_blocks:
        mean = []; std = []
        for n in range(len(lmbdas)):
            data = -pd.read_pickle("result/Convention_LSCGF_N={0}_H={1}_B={2}_λ={3}_lrp={4}.pkl".format(N,n_hid,n_block,lmbdas[n],f2s(lr)))
            std.append(abs(data-ACGF[n]).std())
            mean.append(abs(data-ACGF[n]).mean())
        plt.errorbar(lmbdas,mean,std,capsize=3,fmt="o",ls="-",label='B={0}'.format(n_block))
    # 꾸미기
    plt.ylim(-0.1,1.)
    plt.axhline(y=0,c='black')
    plt.yticks(np.arange(-0.1,1.,0.1))
    plt.legend()
    plt.grid()
    plt.xlabel('λ'); plt.ylabel("$<|\hat{ψ}(λ) - ψ(λ)|>$")
    plt.savefig("_Figure/CGF/N={0}/Difference/H={1}.png".format(N,n_hid),bbox_inches='tight')
    plt.close()
    plt.clf()

N = 21; lr = 1e-3; n_blocks = [1]
n_hids = [25,50,100]
for n_hid in n_hids:
    PlotCGFdiff(N,n_hid,n_blocks,lmbdas,lr)

<Figure size 640x480 with 0 Axes>

### Observable

In [18]:
def ActiveWorkDiff(N,n_hids,n_block,lmbdas,lr):
    # true
    AWT = AV.AW(N,τ,μ,K,Da,kbT,np.array(lmbdas)) 
    # plot
    plt.figure(dpi=400)
    # load data
    for n_hid in n_hids:
        mean = []; std = []
        for n in range(len(lmbdas)):
            data = pd.read_pickle("result/Convention_LWORK_N={0}_H={1}_B={2}_λ={3}_lrp={4}.pkl".format(N,n_hid,n_block,lmbdas[n],f2s(lr)))
            # compute mean and std difference
            std.append(abs(data-AWT[n]).std())
            mean.append(abs(data-AWT[n]).mean())
        plt.errorbar(lmbdas, mean, std, capsize=3, fmt="o", ls="-", label="H = {0}".format(n_hid))
    # 꾸미기
    plt.axhline(0,color="black")
    plt.ylim(-0.01,.2)
    plt.yticks(np.arange(-0.01,.2,0.01))
    plt.grid()
    plt.legend()
    plt.xlabel("λ"); plt.ylabel("$< |\hat{w}(λ) - w(λ)| >$")
    plt.savefig("_Figure/Work/Diff_N={0}.png".format(N),bbox_inches='tight')
    plt.close()
    plt.clf()

N = 21; lr = 1e-3; n_block = 1
n_hids = [25,50]
for n_hid in n_hids:
    ActiveWorkDiff(N,n_hids,n_block,lmbdas,lr)

<Figure size 640x480 with 0 Axes>

### Permutation Error

In [16]:
def PlotError(N,n_hid,n_block,lmbdas,lr):
    epochs = [1000*e for e in range(51)]
    # Relative error
    if not os.path.exists("_Figure/Error/N={0}/".format(N)): os.mkdir("_Figure/Error/N={0}/".format(N))
    for n in range(len(lmbdas)):
        if not os.path.exists("_Figure/Error/N={0}/λ={1}/".format(N,lmbdas[n])): os.mkdir("_Figure/Error/N={0}/λ={1}/".format(N,lmbdas[n])) 
        plt.figure(dpi=400)
        # load data
        data = pd.read_pickle("result/Convention_ERROR_N={0}_H={1}_B={2}_λ={3}_lrp={4}.pkl".format(N,n_hid,n_block,lmbdas[n],f2s(lr)))
        # process data
        pr = []; pa = []
        for e in range(len(epochs)): pr.append(data[e][0]); pa.append(data[e][1])
        # plot
        plt.plot(epochs, pr, alpha=0.7, marker='o', markersize=3, c='red',  label = 'position part')
        plt.plot(epochs, pa, alpha=0.7, marker='o', markersize=3, c='blue', label = 'self propulsion part')
        # 꾸미기
        plt.yscale('log'); plt.xlabel('Epoch')
        plt.grid(True, which='both', linestyle='-', linewidth=0.5)
        plt.minorticks_on()
        plt.savefig("_Figure/Error/N={0}/λ={1}/B={2}_H={3}.png".format(N,lmbdas[n],n_block,n_hid),bbox_inches='tight')
        plt.close()
        plt.clf()

N = 21; lr = 1e-3; n_block = 1
n_hids = [25,50]
for n_hid in n_hids:
    PlotError(N,n_hid,n_block,lmbdas,lr)

<Figure size 640x480 with 0 Axes>

### Large deviation function

In [ ]:
def LDF_analytic(N,tau,mu,K,Da,kbT):
    c = -1/(2*kbT)
    lmbda = np.linspace(c-1.001,c+1.001,1000000)
    AW = AV.AW(N,tau,mu,K,Da,kbT,lmbda); CGF = AV.CGF(N,tau,mu,K,Da,kbT,lmbda); LDF = lmbda*AW-CGF
    return AW,LDF

In [ ]:
# 하나의 N에 대해서 LDF를 그림
def PlotLDF(N,n_hid,n_block,lmbdas,lmbdas2):
    # True
    AAW, ALDF = LDF_analytic(N,τ,μ,K,Da,kbT) # 선
    AAWD = AV.AW(N,τ,μ,K,Da,kbT,np.array(lmbdas2)); ALDFD = np.array(lmbdas2)*AAWD-AV.CGF(N,τ,μ,K,Da,kbT,np.array(lmbdas2)) # 점
    plt.figure(dpi=400)
    # reference
    plt.plot(AAW,ALDF,c='black')
    plt.scatter(AAWD,ALDFD,c='black',marker='o')
    # process & plot
    # load data
    loss = ProcessTrain("SCGF",N,n_hid,n_block,lmbdas); work = ProcessTrain("WORK",N,n_hid,n_block,lmbdas)
    ## data process
    LDF = [0.0]; AW = [AV.AW(N,τ,μ,K,Da,kbT,0.0)]
    for n in range(len(lmbdas)):
        LDF.append(lmbdas[n]*np.array(work[n][-1000:]).mean() - np.array(loss[n][-1000:]).mean())
        AW.append(np.array(work[n][-1000:]).mean())
    plt.plot(AW,LDF,marker="o",markersize=3,ls="-",c='blue')
    # 꾸미기
    # plt.legend(bbox_to_anchor=(1.,1.))
    plt.xlabel("w"); plt.ylabel("$I(w)$")
    plt.xlim(-1+int(min(AW)), int(max(AW))+1)
    plt.ylim(ymax=1+int(max(ALDF)), ymin=0.001)
    if not os.path.exists("_Figure/LDF/N={0}".format(N)): os.mkdir("_Figure/LDF/N={0}".format(N))
    plt.savefig("_Figure/LDF/N={0}/LDF.png".format(N),bbox_inches='tight')
    # log plot
    plt.yscale("log")
    plt.savefig("_Figure/LDF/N={0}/LDF_log.png".format(N),bbox_inches='tight')
    plt.close()
    plt.clf()

Ns = [7, 9, 11]
n_hids = [25, 35, 50]
n_block = 0
for n in range(len(Ns)):
    PlotLDF(Ns[n],n_hids[n],n_block,[-0.25, -0.5, -0.75, -1.0, -1.25, -1.5, -1.75, -2.0], [0.0, -0.25, -0.5, -0.75, -1.0, -1.25, -1.5, -1.75, -2.0])

<Figure size 640x480 with 0 Axes>

### Detailed Flcutuation theorem (train)

In [33]:
def PlotDFT(N,n_hid,n_block,lmbdas):
    # True
    c = -1/(2*kbT); AWs = AV.AW(N,τ,μ,K,Da,kbT,np.linspace(-2.01,0.01,1000000))
    plt.figure(dpi=400)
    plt.plot(AWs,2*c*AWs,color='black',ls=":",label="(True) slope = {0:.2f}".format(2*c))
    LDF = [0.0]; AW = [AV.AW(N,τ,μ,K,Da,kbT,0.0)]
    # load data
    aw  = ProcessTrain("WORK",N,n_hid,n_block,lmbdas)
    cgf = ProcessTrain("SCGF",N,n_hid,n_block,lmbdas)
    # process
    for n in range(len(lmbdas)):
        AW.append(np.array(aw[n][-1000:]).mean())
        LDF.append(lmbdas[n]*np.array(aw[n][-1000:]).mean() - np.array(cgf[n][-1000:]).mean())
    # let's reverse it!
    LDF_flip = np.flip(LDF)
    # slope값 구하기
    fit = linregress(AW,LDF-LDF_flip)
    # plot graph
    plt.scatter(AW,LDF-LDF_flip,marker="o",c='blue',s=5)
    plt.plot(AW,fit.slope*np.array(AW)+fit.intercept,ls="--",c='blue',label="Slope = {0:.2f}".format(fit.slope))
    plt.legend(loc='upper right')
    plt.xlabel("$w$"); plt.ylabel("I(w) - I(-w)")
    if not os.path.exists("_Figure/LDF/N={0}".format(N)): os.mkdir("_Figure/LDF/N={0}".format(N))
    plt.savefig("_Figure/LDF/N={0}/DFT.png".format(N),bbox_inches='tight')
    plt.close()
    plt.clf()

Ns = [7, 9, 11]
n_hids = [25, 35, 50]
n_block = 0
for n in range(len(Ns)):    
    PlotDFT(Ns[n],n_hids[n],n_block,[-0.25, -0.5, -0.75, -1.0, -1.25, -1.5, -1.75, -2.0])

<Figure size 640x480 with 0 Axes>

In [31]:
def PlotDFT_all(Ns, n_hids, n_block, lmbdas):
    cs = [
    '#1f77b4',  # blue
    '#ff7f0e',  # orange
    '#2ca02c',  # green
    '#d62728',  # red
    '#9467bd',  # purple
    '#8c564b',  # brown
    '#e377c2',  # pink
    '#7f7f7f',  # gray
    '#bcbd22',  # yellow-green
    '#17becf'   # cyan
    ]
    # True
    c = -1/(2*kbT); AWs = AV.AW(max(Ns),τ,μ,K,Da,kbT,np.linspace(-2.01,0.01,1000000))
    plt.figure(dpi=400)
    plt.plot(AWs,2*c*AWs,color='black',ls=":",label="(true) slope = {0:.2f}".format(2*c))
    for n in range(len(Ns)):
        LDF = [0.0]; AW = [AV.AW(Ns[n],τ,μ,K,Da,kbT,0.0)]
        # load data
        aw = ProcessTrain("WORK",Ns[n],n_hids[n],n_block,lmbdas)
        cgf = ProcessTrain("SCGF",Ns[n],n_hids[n],n_block,lmbdas)
        # process
        for m in range(len(lmbdas)):
            AW.append(np.array(aw[m][-1000:]).mean())
            LDF.append(lmbdas[m]*np.array(aw[m][-1000:]).mean() - np.array(cgf[m][-1000:]).mean())
        # let's reverse it!
        LDF_flip = np.flip(LDF)
        # slope
        fit = linregress(AW, LDF-LDF_flip)
        # plot
        plt.scatter(AW, LDF-LDF_flip, marker="o", s=5)
        plt.plot(AW, fit.slope*np.array(AW)+fit.intercept, c=cs[n], ls="--", label="N = {0}, slope = {1:.2f}".format(Ns[n], fit.slope))
    plt.xlabel("$w$"); plt.ylabel("I(w) - I(-w)")
    plt.grid()
    plt.legend(loc='upper right')
    plt.savefig("_Figure/LDF/ALLDFT.png",bbox_inches='tight')
    plt.close()
    plt.clf()
PlotDFT_all(Ns, n_hids, n_block, lmbdas)

<Figure size 640x480 with 0 Axes>

### DFT (long)

In [32]:
def PlotDFT_long(N,n_hid,n_block,lmbdas):
    # True
    c = -1/(2*kbT); AWs = AV.AW(N,τ,μ,K,Da,kbT,np.linspace(-2.01,0.01,1000000))
    plt.figure(dpi=400)
    plt.plot(AWs,2*c*AWs,color='black',ls=":",label="(True) Slope = {0:.2f}".format(2*c))
    LDF = [0.0]; AW = [AV.AW(N,τ,μ,K,Da,kbT,0.0)]
    # load data
    for n in range(len(lmbdas)):
        aw = pd.read_pickle("data/Convention_LWORK_N={0}_H={1}_B={2}_λ={3}.pkl".format(N,n_hid,n_block,lmbdas[n]))
        cgf = -pd.read_pickle("data/Convention_LSCGF_N={0}_H={1}_B={2}_λ={3}.pkl".format(N,n_hid,n_block,lmbdas[n]))
        AW.append(np.array(aw).mean())
        LDF.append(lmbdas[n]*np.array(aw).mean() - np.array(cgf).mean())
    # let's reverse it!
    LDF_flip = np.flip(LDF)
    # slope값 구하기
    fit = linregress(AW,LDF-LDF_flip)
    # plot graph
    plt.scatter(AW,LDF-LDF_flip,marker="o",c='blue',s=5)
    plt.plot(AW,fit.slope*np.array(AW)+fit.intercept,ls="--",c='blue',label="Slope = {0:.2f}".format(fit.slope))
    plt.xlabel("$w$"); plt.ylabel("I(w) - I(-w)")
    plt.legend(loc='upper right')
    plt.grid()
    plt.savefig("_Figure/LDF/N={0}/Long_DFT.png".format(N),bbox_inches='tight')
    plt.close()
    plt.clf()

n_hids = [25, 35, 50]; n_block = 0
Ns = [7, 9, 11]
for n in range(len(Ns)):
    PlotDFT_long(Ns[n],n_hids[n],n_block,lmbdas)

<Figure size 640x480 with 0 Axes>

In [34]:
def PlotDFT_all_long(Ns, n_hids, n_block, lmbdas):
    cs = [
    '#1f77b4',  # blue
    '#ff7f0e',  # orange
    '#2ca02c',  # green
    '#d62728',  # red
    '#9467bd',  # purple
    '#8c564b',  # brown
    '#e377c2',  # pink
    '#7f7f7f',  # gray
    '#bcbd22',  # yellow-green
    '#17becf'   # cyan
    ]
    # True
    c = -1/(2*kbT); AWs = AV.AW(max(Ns),τ,μ,K,Da,kbT,np.linspace(-2.01,0.01,1000000))
    plt.figure(dpi=400)
    plt.plot(AWs,2*c*AWs,color='black',ls=":",label="(true) slope = {0:.2f}".format(2*c))
    for n in range(len(Ns)):
        LDF = [0.0]; AW = [AV.AW(Ns[n],τ,μ,K,Da,kbT,0.0)]
        # load data
        for m in range(len(lmbdas)):
            aw = pd.read_pickle("data/Convention_LWORK_N={0}_H={1}_B={2}_λ={3}.pkl".format(Ns[n],n_hids[n],n_block,lmbdas[m]))
            cgf = -pd.read_pickle("data/Convention_LSCGF_N={0}_H={1}_B={2}_λ={3}.pkl".format(Ns[n],n_hids[n],n_block,lmbdas[m]))
            AW.append(np.array(aw).mean())
            LDF.append(lmbdas[m]*np.array(aw).mean() - np.array(cgf).mean())
        # let's reverse it!
        LDF_flip = np.flip(LDF)
        # slope
        fit = linregress(AW, LDF-LDF_flip)
        # plot
        plt.scatter(AW, LDF-LDF_flip, marker="o", s=5)
        plt.plot(AW, fit.slope*np.array(AW)+fit.intercept, c=cs[n], ls="--", label="N = {0}, slope = {1:.2f}".format(Ns[n], fit.slope))
    plt.xlabel("$w$"); plt.ylabel("I(w) - I(-w)")
    plt.grid()
    plt.legend(loc='upper right')
    plt.savefig("_Figure/LDF/Long_ALLDFT.png",bbox_inches='tight')
    plt.close()
    plt.clf()

PlotDFT_all_long(Ns, n_hids, 0, lmbdas)

<Figure size 640x480 with 0 Axes>